# Knowledge Graph: entity & relationship extraction

Flow: **load chunks → schema → prompt → extract one chunk → normalize names → validate → test on 15 chunks → review**

What changed after the first test (5 AGI chunks, only ~3 of 19 relations were correct):

| Problem in v1 | Fix here |
|---|---|
| `IS_A` / `RELATED_TO` / `INTRODUCES` used for everything ("GPT-4 IS_A mathematics") | Removed the vague types. Added `HAS_CAPABILITY`, `OUTPERFORMS`, `DEVELOPED_BY`, `EVALUATED_ON`… each with a definition and direction |
| Fields typed as DATASET, authors typed as ORGANIZATION | New `DOMAIN` type. Front matter, acknowledgments and author lists are skipped |
| "Artificial intelligence" vs "artificial intelligence", LLM vs large language model | `aliases` field + one alias map + case-insensitive names |
| Correct relations rejected because of `GPT--4` or curly quotes | Quotes and dashes are normalized before matching |
| Wrong relations accepted because the quote existed somewhere | **Both entity names must appear inside the quote**, type rules (e.g. `DEVELOPED_BY` must point to an ORGANIZATION), one quote can't back more than 3 relations |
| Test only covered 5 AGI chunks | Test set = 3 chunks from **each** of the 5 papers |

**v3 changes** (after the v2 test: 6 of 14 relations correct, 43%):

| Problem in v2 | Fix in v3 |
|---|---|
| Citations became entities ("Mei et al.", "Devlin et al. 2019") | Entities with `et al.` or `[...]` are dropped, and the prompt says so |
| Good relations rejected: the quote skipped "GPT-4" or dropped `[362]` | Citation markers removed before matching; names are checked in the **whole sentence** around the quote |
| "max 3 relations per quote" killed correct ones (GPT-4 IS_A LLM) | Rule removed. Instead: only one relation per entity pair per chunk |
| HELM "DEVELOPED_BY" language models, misuse risks called limitations | New `EVALUATES` and `HAS_RISK` types; `PART_OF` tightened; ORGANIZATION only allowed in `DEVELOPED_BY` |
| "examples include BERT, GPT-3, CLIP" gave nothing | New prompt example for that pattern |

## 1. Setup and load chunks

In [ ]:
import json
import re
import time
from pathlib import Path
from collections import Counter
from typing import Literal

import requests
import pandas as pd
from pydantic import BaseModel, Field

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CHUNKS_PATH = PROJECT_ROOT / "data" / "processed" / "chunks.jsonl"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL_NAME = "qwen3:4b-instruct"

with CHUNKS_PATH.open(encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f if line.strip()]
chunk_by_id = {c["chunk_id"]: c for c in chunks}

In [ ]:
# check: chunks loaded
print("chunks:", len(chunks))
print(Counter(c["document"] for c in chunks))
print("fields:", list(chunks[0].keys()))

In [ ]:
# check: Ollama is running and the model is installed
tags = requests.get("http://localhost:11434/api/tags", timeout=10).json()
installed = [m["name"] for m in tags.get("models", [])]
print("installed:", installed)
assert MODEL_NAME in installed, f"{MODEL_NAME} not installed. Run: ollama pull {MODEL_NAME}"

## 2. Schema: what the model is allowed to return

**Entity types**: what kind of thing a name is.
**Relation types**: each has a fixed **direction** (`source → target`) and a one-line meaning. The meaning goes into the prompt.
There's deliberately **no "RELATED_TO"**: if none of these fits, the model should return nothing.

In [ ]:
ENTITY_TYPES = {
    "MODEL":        "a specific AI model or system, e.g. GPT-4, PaLM, BERT",
    "ORGANIZATION": "a company, lab or university, e.g. OpenAI, Google, Stanford",
    "BENCHMARK":    "an evaluation suite or test, e.g. HELM, MMLU, HumanEval",
    "DATASET":      "a collection of data used for training, e.g. Common Crawl, The Pile",
    "METHOD":       "a technique or algorithm, e.g. RLHF, chain-of-thought prompting, transformer",
    "METRIC":       "something measured, e.g. accuracy, calibration, toxicity",
    "TASK":         "a concrete task, e.g. question answering, summarization",
    "DOMAIN":       "a field or subject area, e.g. medicine, law, mathematics, coding",
    "CONCEPT":      "an abstract idea, e.g. artificial general intelligence, homogenization, hallucination",
}

RELATION_TYPES = {
    "DEVELOPED_BY":   "MODEL/BENCHMARK -> ORGANIZATION that built it",
    "IS_A":           "specific thing -> more general category (GPT-4 IS_A large language model)",
    "OUTPERFORMS":    "MODEL/METHOD -> MODEL/METHOD it is explicitly said to beat",
    "EVALUATES":      "BENCHMARK -> MODEL it tests (HELM EVALUATES GPT-3)",
    "COMPARED_WITH":  "A -> B when the text compares them without a clear winner",
    "EVALUATED_ON":   "MODEL -> BENCHMARK/DATASET/TASK it was tested on",
    "MEASURES":       "BENCHMARK/METRIC -> METRIC/CONCEPT it measures (HELM MEASURES calibration)",
    "TRAINED_ON":     "MODEL -> DATASET it was trained on",
    "USES":           "MODEL/METHOD -> METHOD it uses (GPT-4 USES RLHF)",
    "BUILDS_ON":      "newer thing -> older thing it extends",
    "PART_OF":        "component -> the whole it belongs to (a dataset in a benchmark). NOT for two things listed together",
    "HAS_CAPABILITY": "MODEL -> TASK/DOMAIN it is said to be able to do",
    "HAS_LIMITATION": "MODEL/METHOD -> CONCEPT describing its own weakness (GPT-4 HAS_LIMITATION hallucination)",
    "HAS_RISK":       "MODEL/METHOD -> CONCEPT it could be misused for or cause (GPT-4 HAS_RISK disinformation)",
}

# simple sanity rules: (relation, which end) -> allowed entity types
TYPE_RULES = {
    ("DEVELOPED_BY", "target"):   {"ORGANIZATION"},
    ("TRAINED_ON", "target"):     {"DATASET"},
    ("EVALUATED_ON", "target"):   {"BENCHMARK", "DATASET", "TASK"},
    ("HAS_CAPABILITY", "source"): {"MODEL", "METHOD"},
    ("HAS_CAPABILITY", "target"): {"TASK", "DOMAIN"},
    ("OUTPERFORMS", "source"):    {"MODEL", "METHOD"},
    ("EVALUATES", "source"):      {"BENCHMARK"},
    ("EVALUATES", "target"):      {"MODEL"},
    ("HAS_LIMITATION", "source"): {"MODEL", "METHOD"},
    ("HAS_RISK", "source"):       {"MODEL", "METHOD"},
}

In [ ]:
class Entity(BaseModel):
    name: str = Field(description="Full name, singular, as written in the passage (no acronym if the full name is given)")
    entity_type: Literal["MODEL", "ORGANIZATION", "BENCHMARK", "DATASET", "METHOD",
                         "METRIC", "TASK", "DOMAIN", "CONCEPT"]
    aliases: list[str] = Field(default_factory=list, description="Other names used in the passage, e.g. acronyms like LLM, AGI")


class Relationship(BaseModel):
    source: str = Field(description="Entity name (must be one of the extracted entities)")
    relation: Literal["DEVELOPED_BY", "IS_A", "OUTPERFORMS", "EVALUATES", "COMPARED_WITH", "EVALUATED_ON",
                      "MEASURES", "TRAINED_ON", "USES", "BUILDS_ON", "PART_OF",
                      "HAS_CAPABILITY", "HAS_LIMITATION", "HAS_RISK"]
    target: str = Field(description="Entity name (must be one of the extracted entities)")
    evidence: str = Field(description="ONE sentence or clause copied exactly from the passage that states this relation")


class ChunkExtraction(BaseModel):
    entities: list[Entity]
    relationships: list[Relationship]

In [ ]:
# check: schema matches the dictionaries above
assert set(Entity.model_fields["entity_type"].annotation.__args__) == set(ENTITY_TYPES)
assert set(Relationship.model_fields["relation"].annotation.__args__) == set(RELATION_TYPES)
display(pd.DataFrame([{"relation": k, "meaning": v} for k, v in RELATION_TYPES.items()]))

## 3. Prompt with examples

Small models follow **examples** much better than rules, so the prompt shows four:
1. a passage with good relations,
2. a passage where the right answer is a **list of capabilities** (not `IS_A`),
3. "examples include X, Y" → `IS_A`, a benchmark that `EVALUATES` a model, a misuse risk, and two traps: citations and "X and Y" lists,
4. a passage with **no** relations at all.

In [ ]:
EXAMPLES = """
EXAMPLE 1
Passage: The latest model developed by OpenAI, GPT-4, was trained using an unprecedented scale of compute and data. GPT-4 is part of a new cohort of LLMs, along with ChatGPT, and often vastly surpasses prior models such as ChatGPT.
Output:
{"entities": [
  {"name": "GPT-4", "entity_type": "MODEL", "aliases": []},
  {"name": "OpenAI", "entity_type": "ORGANIZATION", "aliases": []},
  {"name": "large language model", "entity_type": "CONCEPT", "aliases": ["LLM"]},
  {"name": "ChatGPT", "entity_type": "MODEL", "aliases": []}],
 "relationships": [
  {"source": "GPT-4", "relation": "DEVELOPED_BY", "target": "OpenAI", "evidence": "The latest model developed by OpenAI, GPT-4"},
  {"source": "GPT-4", "relation": "IS_A", "target": "large language model", "evidence": "GPT-4 is part of a new cohort of LLMs"},
  {"source": "GPT-4", "relation": "OUTPERFORMS", "target": "ChatGPT", "evidence": "often vastly surpasses prior models such as ChatGPT"}]}

EXAMPLE 2
Passage: GPT-4 can solve novel and difficult tasks that span mathematics, coding and medicine. HELM measures calibration and robustness for every model it evaluates.
Output:
{"entities": [
  {"name": "GPT-4", "entity_type": "MODEL", "aliases": []},
  {"name": "mathematics", "entity_type": "DOMAIN", "aliases": []},
  {"name": "coding", "entity_type": "DOMAIN", "aliases": []},
  {"name": "medicine", "entity_type": "DOMAIN", "aliases": []},
  {"name": "Holistic Evaluation of Language Models", "entity_type": "BENCHMARK", "aliases": ["HELM"]},
  {"name": "calibration", "entity_type": "METRIC", "aliases": []},
  {"name": "robustness", "entity_type": "METRIC", "aliases": []}],
 "relationships": [
  {"source": "GPT-4", "relation": "HAS_CAPABILITY", "target": "mathematics", "evidence": "GPT-4 can solve novel and difficult tasks that span mathematics"},
  {"source": "Holistic Evaluation of Language Models", "relation": "MEASURES", "target": "calibration", "evidence": "HELM measures calibration"},
  {"source": "Holistic Evaluation of Language Models", "relation": "MEASURES", "target": "robustness", "evidence": "HELM measures calibration and robustness"}]}

EXAMPLE 3
Passage: A foundation model is any model trained on broad data; current examples include BERT [Devlin et al. 2019] and GPT-3 [Brown et al. 2020]. HELM evaluates 30 language models, including GPT-3. Models like GPT-4 can be misused to generate disinformation. Healthcare and biomedicine are large application areas.
Output:
{"entities": [
  {"name": "foundation model", "entity_type": "CONCEPT", "aliases": []},
  {"name": "BERT", "entity_type": "MODEL", "aliases": []},
  {"name": "GPT-3", "entity_type": "MODEL", "aliases": []},
  {"name": "Holistic Evaluation of Language Models", "entity_type": "BENCHMARK", "aliases": ["HELM"]},
  {"name": "GPT-4", "entity_type": "MODEL", "aliases": []},
  {"name": "disinformation", "entity_type": "CONCEPT", "aliases": []},
  {"name": "healthcare", "entity_type": "DOMAIN", "aliases": []},
  {"name": "biomedicine", "entity_type": "DOMAIN", "aliases": []}],
 "relationships": [
  {"source": "BERT", "relation": "IS_A", "target": "foundation model", "evidence": "A foundation model is any model trained on broad data; current examples include BERT"},
  {"source": "GPT-3", "relation": "IS_A", "target": "foundation model", "evidence": "A foundation model is any model trained on broad data; current examples include BERT [Devlin et al. 2019] and GPT-3"},
  {"source": "Holistic Evaluation of Language Models", "relation": "EVALUATES", "target": "GPT-3", "evidence": "HELM evaluates 30 language models, including GPT-3"},
  {"source": "GPT-4", "relation": "HAS_RISK", "target": "disinformation", "evidence": "Models like GPT-4 can be misused to generate disinformation"}]}
(Note: no entity for "Devlin et al." or "Brown et al." - they are citations. No relation between healthcare and biomedicine - being listed together is not a relation.)

EXAMPLE 4
Passage: Intelligence is a multifaceted and elusive concept that has long challenged psychologists, philosophers, and computer scientists.
Output:
{"entities": [], "relationships": []}
"""

SYSTEM_PROMPT = f"""You extract a knowledge graph from passages of AI/ML research papers.

ENTITY TYPES:
{chr(10).join(f"- {k}: {v}" for k, v in ENTITY_TYPES.items())}

RELATION TYPES (direction is source -> target):
{chr(10).join(f"- {k}: {v}" for k, v in RELATION_TYPES.items())}

RULES:
1. Use ONLY the passage. No outside knowledge.
2. Extract specific, meaningful entities. NEVER make an entity from a person, an author or a citation
   ("Devlin et al.", "Mei et al. (2021)", "[Ope23]"). Skip generic words (system, paper, results, tasks).
3. Entity names: full name, singular, as written. Put acronyms in aliases (LLM, AGI, HELM).
4. Only add a relation when the passage STATES it. Two names in the same sentence is NOT enough.
5. If no relation type fits exactly, leave the relation out. Empty lists are fine.
   PART_OF is only for a component inside a larger whole. Things listed together ("X and Y") are NOT related.
   ORGANIZATION entities are only used with DEVELOPED_BY.
6. evidence = one short sentence or clause copied exactly from the passage, containing BOTH entity names (or their aliases).
7. source and target must be names from your entities list.

{EXAMPLES}"""

print(SYSTEM_PROMPT[:1500], "...")

## 4. Extract one chunk

`format=` passes the JSON schema to Ollama, so the answer always has the right shape. `temperature=0` makes it repeatable.

In [ ]:
def extract_chunk(chunk):
    user_prompt = f"""Paper: {chunk["title"]}
Section: {" > ".join(chunk["section_path"])}

Passage: {chunk["text"]}
Output:"""

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL_NAME,
            "stream": False,
            "format": ChunkExtraction.model_json_schema(),
            "options": {"temperature": 0, "num_ctx": 8192},
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
        },
        timeout=300,
    )
    response.raise_for_status()
    return ChunkExtraction.model_validate_json(response.json()["message"]["content"])

In [ ]:
# check: run on ONE chunk and look at the raw result
sample_chunk = chunk_by_id["GPT-4_0011"] if "GPT-4_0011" in chunk_by_id else chunks[10]
print(sample_chunk["chunk_id"], "|", " > ".join(sample_chunk["section_path"]))
print(sample_chunk["text"][:600], "\n")

start = time.time()
sample_extraction = extract_chunk(sample_chunk)
print(f"took {time.time() - start:.1f}s\n")

for e in sample_extraction.entities:
    print(f"  ENTITY  {e.name:40} {e.entity_type:12} {e.aliases}")
for r in sample_extraction.relationships:
    print(f"  REL     {r.source} --{r.relation}--> {r.target}\n          \"{r.evidence}\"")

## 5. Normalize names

The same thing gets written many ways (`LLMs`, `large language models`, `Large language model`).
`canonical()` turns every spelling into **one** name:
1. tidy up the text: spaces, `GPT--4` → `GPT-4`
2. look it up in `ALIASES` (lowercase keys)
3. otherwise keep the name as written. Matching is case-insensitive, so "Artificial intelligence" = "artificial intelligence".

`ALIASES` starts with a few known names. The model's `aliases` field adds more for each chunk.

In [ ]:
ALIASES = {
    "llm": "large language model",
    "llms": "large language model",
    "large language models": "large language model",
    "language models": "language model",
    "lm": "language model",
    "agi": "artificial general intelligence",
    "ai": "artificial intelligence",
    "helm": "Holistic Evaluation of Language Models",
    "foundation models": "foundation model",
    "gpt4": "GPT-4",
    "gpt-4 model": "GPT-4",
    "rlhf": "reinforcement learning from human feedback",
    "cot": "chain-of-thought prompting",
}


def tidy(text):
    text = text.replace("“", '"').replace("”", '"').replace("’", "'").replace("‘", "'")
    text = text.replace("--", "-").replace("–", "-").replace("—", "-")
    return re.sub(r"\s+", " ", text).strip()


def canonical(name, extra_aliases=None):
    name = tidy(name)
    key = name.lower()
    if extra_aliases and key in extra_aliases:
        return extra_aliases[key]
    return ALIASES.get(key, name)


def key_of(name):
    """Case-insensitive key used to compare names."""
    return canonical(name).lower()

In [ ]:
# check: different spellings should give the same key
tests = ["LLMs", "large language models", "Large language model", "GPT--4", "gpt-4",
         "HELM", "Holistic Evaluation of Language Models", "AGI", "Artificial General Intelligence"]
for t in tests:
    print(f"{t:42} -> {key_of(t)}")

## 6. Validate every relationship

A relationship is **accepted** only if all checks pass. Otherwise it's **rejected** with a reason, so you can see *why*.

| Check | Catches |
|---|---|
| entity name appears in the passage (otherwise the entity is dropped) | invented entities |
| source and target are extracted entities | made-up endpoints like "early writings" |
| source ≠ target | self-loops |
| entity is not a citation ("et al.", "[..]") | authors and references as entities |
| quote found in the passage (after tidying quotes, dashes and **removing citation markers** like `[362]`) | invented quotes |
| at least one name is **in the quote**, and **both** are in the quote's sentence | quotes that don't talk about the pair, while still allowing "GPT-4 … passing a simulated bar exam" |
| type rules (`DEVELOPED_BY` → ORGANIZATION, `EVALUATES` → MODEL, ORGANIZATION only in `DEVELOPED_BY`…) | "GPT-4 DEVELOPED_BY mathematics", "Restaurant PART_OF Mei et al." |
| one relation per entity pair per chunk | "GPT-4 IS_A LLM" + "GPT-4 PART_OF LLM" duplicates |

In [ ]:
CITATION = re.compile(r"\[[^\]]{1,40}\]"                       # [362], [Ope23], [Devlin et al. 2019]
                      r"|\((?:[^()]*et al\.?[^()]*|[^()]*(?:19|20)\d\d[a-z]?)\)")   # (Konda et al., 2016), (2021)


def for_matching(text):
    """Text used for quote matching: tidy, no citation markers, lowercase."""
    text = CITATION.sub("", tidy(text))
    return re.sub(r"\s+", " ", text).strip().lower()


def is_citation(name):
    return "et al" in name.lower() or bool(re.search(r"\[|\]", name))


def sentence_around(passage, start, end):
    """The full sentence(s) that contain passage[start:end]."""
    left = max(passage.rfind(". ", 0, start), passage.rfind("? ", 0, start), passage.rfind("! ", 0, start))
    left = 0 if left == -1 else left + 2
    ends = [i for i in (passage.find(". ", end), passage.find("? ", end), passage.find("! ", end)) if i != -1]
    right = min(ends) + 1 if ends else len(passage)
    return passage[left:right]


def mentions(quote_key, entity_name, entity_aliases):
    """True if the entity's name, an alias, or a known ALIASES spelling appears in the quote
    as a whole word (an optional plural "s" is allowed, so "LLMs" matches "LLM")."""
    names = [entity_name] + entity_aliases
    names += [k for k, v in ALIASES.items() if v.lower() == entity_name.lower()]
    for n in names:
        pattern = r"(?<![a-z0-9])" + re.escape(tidy(n).lower()) + r"s?(?![a-z0-9])"
        if n and re.search(pattern, quote_key):
            return True
    return False


def validate(chunk, extraction):
    passage_key = for_matching(chunk["text"])

    # entity lookup: canonical key -> entity info (with aliases from this chunk)
    chunk_aliases = {}
    for e in extraction.entities:
        for a in e.aliases:
            chunk_aliases[tidy(a).lower()] = canonical(e.name)
    entities = {}
    for e in extraction.entities:
        if is_citation(e.name):                                # "Devlin et al." is not an entity
            continue
        if not mentions(tidy(chunk["text"]).lower(), e.name, e.aliases):   # entity must really be in the passage
            continue
        name = canonical(e.name, chunk_aliases)
        entities[name.lower()] = {"name": name, "type": e.entity_type,
                                  "aliases": [e.name] + e.aliases}

    seen_pairs = set()
    accepted, rejected = [], []
    for r in extraction.relationships:
        src = entities.get(canonical(r.source, chunk_aliases).lower())
        tgt = entities.get(canonical(r.target, chunk_aliases).lower())
        quote_key = for_matching(r.evidence)
        position = passage_key.find(quote_key) if quote_key else -1
        sentence = sentence_around(passage_key, position, position + len(quote_key)) if position >= 0 else ""

        if src is None:
            reason = "source not in entities"
        elif tgt is None:
            reason = "target not in entities"
        elif src["name"] == tgt["name"]:
            reason = "source == target"
        elif not quote_key:
            reason = "empty quote"
        elif position < 0:
            reason = "quote not in passage"
        elif not (mentions(quote_key, src["name"], src["aliases"]) or mentions(quote_key, tgt["name"], tgt["aliases"])):
            reason = "neither entity is in the quote"
        elif not mentions(sentence, src["name"], src["aliases"]):
            reason = "source not in the quote's sentence"
        elif not mentions(sentence, tgt["name"], tgt["aliases"]):
            reason = "target not in the quote's sentence"
        elif "ORGANIZATION" in (src["type"], tgt["type"]) and r.relation != "DEVELOPED_BY":
            reason = f"ORGANIZATION only allowed in DEVELOPED_BY, not {r.relation}"
        elif src["type"] not in TYPE_RULES.get((r.relation, "source"), {src["type"]}):
            reason = f"source type {src['type']} not allowed for {r.relation}"
        elif tgt["type"] not in TYPE_RULES.get((r.relation, "target"), {tgt["type"]}):
            reason = f"target type {tgt['type']} not allowed for {r.relation}"
        elif (src["name"], tgt["name"]) in seen_pairs:
            reason = "duplicate: this pair already has a relation in this chunk"
        else:
            reason = None
            seen_pairs.add((src["name"], tgt["name"]))

        row = {
            "source": src["name"] if src else r.source,
            "relation": r.relation,
            "target": tgt["name"] if tgt else r.target,
            "evidence": r.evidence,
            "chunk_id": chunk["chunk_id"],
            "document": chunk["document"],
            "section": " > ".join(chunk["section_path"]),
            "page": chunk["page_start"],
        }
        if reason:
            rejected.append({**row, "reason": reason})
        else:
            accepted.append(row)

    entity_rows = [{"name": v["name"], "type": v["type"], "chunk_id": chunk["chunk_id"],
                    "document": chunk["document"]} for v in entities.values()]
    return entity_rows, accepted, rejected

In [ ]:
# check: validation on a hand-made example (no LLM needed)
fake_chunk = {"chunk_id": "demo", "document": "demo", "section_path": ["demo"], "page_start": 1,
              "text": "The latest model developed by OpenAI, GPT-4, beats ChatGPT. "
                      "GPT-4 can solve tasks that span mathematics, coding, vision, medicine and law. "
                      "We use the Beer dataset from Mei et al. (2021) [12]."}
fake = ChunkExtraction(
    entities=[Entity(name="GPT-4", entity_type="MODEL"), Entity(name="OpenAI", entity_type="ORGANIZATION"),
              Entity(name="ChatGPT", entity_type="MODEL"), Entity(name="mathematics", entity_type="DOMAIN"),
              Entity(name="coding", entity_type="DOMAIN"), Entity(name="vision", entity_type="DOMAIN"),
              Entity(name="medicine", entity_type="DOMAIN"), Entity(name="Beer", entity_type="DATASET"),
              Entity(name="Mei et al.", entity_type="ORGANIZATION")],
    relationships=[
        Relationship(source="GPT--4", relation="DEVELOPED_BY", target="OpenAI",
                     evidence="model developed by OpenAI, GPT--4"),                    # ok (dash fixed)
        Relationship(source="GPT-4", relation="OUTPERFORMS", target="ChatGPT",
                     evidence="The latest model developed by OpenAI"),               # neither name in quote
        Relationship(source="GPT-4", relation="DEVELOPED_BY", target="ChatGPT",
                     evidence="GPT-4, beats ChatGPT"),                               # wrong type
        Relationship(source="GPT-4", relation="HAS_CAPABILITY", target="law",
                     evidence="span mathematics, coding, vision, medicine and law"), # law not an entity
        Relationship(source="GPT-4", relation="HAS_CAPABILITY", target="mathematics",
                     evidence="span mathematics, coding"),                           # ok: GPT-4 is in the sentence
        Relationship(source="GPT-4", relation="IS_A", target="mathematics",
                     evidence="span mathematics, coding"),                           # duplicate pair
        Relationship(source="Beer", relation="PART_OF", target="Mei et al.",
                     evidence="the Beer dataset from Mei et al. (2021)"),           # citation entity dropped
        Relationship(source="GPT-4", relation="HAS_CAPABILITY", target="coding",
                     evidence="We use the Beer dataset from Mei et al. [12]"),     # quote found (citations ignored), but GPT-4 is not in that sentence
    ],
)
_, ok, bad = validate(fake_chunk, fake)
print("accepted:"); display(pd.DataFrame(ok)[["source", "relation", "target"]])
print("rejected:"); display(pd.DataFrame(bad)[["source", "relation", "target", "reason"]])

## 7. Pick the test set: 3 chunks from each paper

We skip sections with no useful relations (front matter, acknowledgments, author lists, prompt-example appendices).
Then we take chunks that mention our key entities (GPT-4, HELM, foundation model…), spread out over each paper.

In [ ]:
SKIP_WORDS = ["front matter", "acknowledg", "author contributions", "rbrm instructions",
              "table full examples", "table examples", "prompt 1", "system card"]
KEYWORDS = ["gpt-4", "helm", "foundation model", "calibration", "openai", "benchmark", "outperform"]
PER_DOCUMENT = 3


def is_useful(chunk):
    section = " > ".join(chunk["section_path"]).lower()
    return not any(word in section for word in SKIP_WORDS)


test_chunks = []
for document in sorted({c["document"] for c in chunks}):
    candidates = [c for c in chunks
                  if c["document"] == document and is_useful(c)
                  and any(k in c["text"].lower() for k in KEYWORDS)]
    step = max(1, len(candidates) // PER_DOCUMENT)
    test_chunks += candidates[::step][:PER_DOCUMENT]

In [ ]:
# check: which chunks are in the test set?
display(pd.DataFrame([{
    "chunk_id": c["chunk_id"],
    "section": " > ".join(c["section_path"])[:60],
    "keywords": [k for k in KEYWORDS if k in c["text"].lower()],
} for c in test_chunks]))

## 8. Run extraction with checkpointing

Each chunk's result is **appended to a `.jsonl` file as soon as it's done**. If the run stops (crash, closed laptop, Ollama error), just run the cell again: it **skips chunks already in the file** and **retries chunks that errored**.
This same function will process all ~2,900 chunks later; only the list and the file name change.

In [ ]:
def load_done(path):
    if not path.exists():
        return {}
    with path.open(encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]
    return {row["chunk_id"]: row for row in rows}


def run_extraction(chunk_list, output_path):
    done = load_done(output_path)
    # skip chunks that are done; retry the ones that errored last time
    todo = [c for c in chunk_list if c["chunk_id"] not in done or done[c["chunk_id"]]["error"]]
    print(f"{len(chunk_list) - len(todo)} already done, {len(todo)} to go")

    for i, chunk in enumerate(todo, start=1):
        start = time.time()
        try:
            extraction = extract_chunk(chunk)
            entity_rows, accepted, rejected = validate(chunk, extraction)
            row = {"chunk_id": chunk["chunk_id"], "entities": entity_rows,
                   "accepted": accepted, "rejected": rejected, "error": None}
        except Exception as exc:
            row = {"chunk_id": chunk["chunk_id"], "entities": [],
                   "accepted": [], "rejected": [], "error": str(exc)}

        with output_path.open("a", encoding="utf-8") as f:     # checkpoint: save right away
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

        print(f"[{i}/{len(todo)}] {chunk['chunk_id']:25} "
              f"entities={len(row['entities']):2} accepted={len(row['accepted']):2} "
              f"rejected={len(row['rejected']):2} {time.time() - start:5.1f}s"
              + (f"  ERROR: {row['error'][:80]}" if row["error"] else ""))

    return load_done(output_path)

In [ ]:
TEST_OUTPUT = ARTIFACTS_DIR / "extractions_test_v3.jsonl"
test_results = run_extraction(test_chunks, TEST_OUTPUT)

## 9. Review the results

Three checks, matching the three questions:
1. **Entity quality**: types and names, and whether any name got two different types.
2. **Relationship quality**: accepted relations with their quotes. Read them!
3. **Evidence quality**: why things were rejected.

In [ ]:
results = [test_results[c["chunk_id"]] for c in test_chunks if c["chunk_id"] in test_results]
all_entities = [e for r in results for e in r["entities"]]
all_accepted = [a for r in results for a in r["accepted"]]
all_rejected = [x for r in results for x in r["rejected"]]

print("chunks:", len(results), "| errors:", sum(r["error"] is not None for r in results))
print("entities:", len(all_entities), "| accepted relations:", len(all_accepted),
      "| rejected relations:", len(all_rejected))
print("\naccepted by relation type:", dict(Counter(a["relation"] for a in all_accepted)))
print("rejected by reason:", dict(Counter(x["reason"] for x in all_rejected)))

In [ ]:
# check 1: entity quality
entities_df = pd.DataFrame(all_entities, columns=["name", "type", "chunk_id", "document"])
display(entities_df.groupby(["type"]).size().rename("count").to_frame().T)

summary = entities_df.groupby("name").agg(types=("type", lambda t: sorted(set(t))),
                                          mentions=("type", "size"))
print("names with MORE THAN ONE type (should be fixed or added to ALIASES):")
display(summary[summary["types"].map(len) > 1])
print("most frequent entities:")
display(summary.sort_values("mentions", ascending=False).head(15))

In [ ]:
# check 2: relationship quality - read every accepted relation
pd.set_option("display.max_colwidth", 120)
display(pd.DataFrame(all_accepted, columns=["source", "relation", "target", "evidence", "chunk_id"]))

In [ ]:
# check 3: evidence quality - what got rejected, and why
display(pd.DataFrame(all_rejected, columns=["source", "relation", "target", "reason", "evidence"]))

## 10. Grade it by hand

The checks can't tell whether a relation is *true*, only whether it's well-formed. So:
1. Run the cell below. It writes `artifacts/review_test_v3.csv` with an empty `verdict` column.
2. Open it in Excel and fill `verdict` with `correct`, `partial` or `wrong` for each row.
3. Run the next cell to get the precision.

Aim for **≥ 80% correct** before running all chunks. If it's lower, look at the wrong rows: they tell you which prompt example, relation definition or alias to change.

In [ ]:
REVIEW_PATH = ARTIFACTS_DIR / "review_test_v3.csv"

review = pd.DataFrame(all_accepted, columns=["chunk_id", "source", "relation", "target", "evidence"])
review["verdict"] = ""
if not REVIEW_PATH.exists():               # never overwrite your grading
    review.to_csv(REVIEW_PATH, index=False, encoding="utf-8-sig")
    print("written:", REVIEW_PATH)
else:
    print("already exists (not overwritten):", REVIEW_PATH)

In [ ]:
# check: precision from your grading
graded = pd.read_csv(REVIEW_PATH)
graded["verdict"] = graded["verdict"].fillna("").str.strip().str.lower()
done = graded[graded["verdict"] != ""]

print(f"graded {len(done)} of {len(graded)}")
if len(done):
    counts = done["verdict"].value_counts()
    print(counts.to_string())
    print(f"\nprecision (correct only): {counts.get('correct', 0) / len(done):.0%}")
    print(f"precision (correct + partial): {(counts.get('correct', 0) + counts.get('partial', 0)) / len(done):.0%}")
    display(done[done["verdict"] == "wrong"][["source", "relation", "target", "evidence"]])

### Next
When the test looks good:
1. Run `run_extraction(all useful chunks, ARTIFACTS_DIR / "extractions_all.jsonl")`. It's resumable, so it's fine to stop and restart.
2. Merge duplicate entities across papers (`canonical()` + more `ALIASES`).
3. Build the NetworkX graph from the accepted relations.